In [ ]:
import os
from pathlib import Path
import pandas as pd
import requests
import numpy as np
from dotenv import load_dotenv
import time
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
from datetime import datetime, timedelta
import yfinance as yf

# Descargar recursos de NLTK si no existen
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download("vader_lexicon")

In [ ]:
# -----------------------------
# CONFIGURACIÓN DEL PROYECTO
# -----------------------------
ROOT_DIR = Path.cwd().parent
ENV_PATH = ROOT_DIR / ".env"
load_dotenv(ENV_PATH)


DATA_RAW = ROOT_DIR / "data/raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_yfinance_history(ticker, start, end):
    """Descarga 20 años de historia diaria usando Yahoo Finance"""
    print(f"   ⬇ Descargando YFinance para {ticker}...")
    try:
        # Descarga
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        
        # Aplanar MultiIndex si existe (corrección común en versiones recientes de yfinance)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
            
        df = df.reset_index()
        
        # Renombrar columnas a minúsculas estándar
        df = df.rename(columns={
            "Date": "timestamp",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "volume"
        })
        
        # Asegurar formato fecha
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        return df
    except Exception as e:
        print(f"   ⚠ Error con YFinance: {e}")
        return pd.DataFrame()

In [ ]:
def add_derived_features(df):
    """Calcula indicadores técnicos sobre el precio de cierre principal"""
    df = df.sort_values("timestamp").copy()
    
    # Usamos 'close' (que vendrá de YFinance)
    df["return"] = df["close"].pct_change()
    
    # Volatilidad (Rolling 20 días)
    df["volatility"] = df["return"].rolling(window=20).std()
    
    # RSI (14 días)
    delta = df["close"].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-9)
    df["rsi"] = 100 - (100 / (1 + rs))
    
    # Rangos
    df["hl_range"] = df["high"] - df["low"]
    df["oc_range"] = df["open"] - df["close"]
    
    # Medias Móviles (Simple)
    df["ma_50"] = df["close"].rolling(50).mean()
    df["ma_200"] = df["close"].rolling(200).mean() # Importante para largo plazo
    
    # Log Volume
    df["log_volume"] = np.log1p(df["volume"])
    
    return df

In [ ]:
ticker = "NVDA"
end_date = pd.Timestamp.today()
start_date = end_date - pd.DateOffset(years=20) # 20 AÑOS

print(f"🚀 Iniciando proceso para {ticker} desde {start_date.date()} hasta {end_date.date()}...")

# A. DESCARGAS
df_yf = get_yfinance_history(ticker, start_date, end_date)
# B. MERGE (FUSIÓN)
# Usamos YFinance como esqueleto base (fechas completas)
df_final = df_yf.copy()

# 1. Unir Polygon (Solo nos interesa VWAP y Num_Trades si existen, el precio OHLC ya lo tenemos de YF)
df_final["vwap"] = df_final["close"] # Fallback
df_final["num_trades"] = 0

# 2. Unir Earnings
df_final["surprisePercent"] = np.nan

# C. TRATAMIENTO Y LIMPIEZA DE DATOS (Data Cleaning)
print(" Tratando y limpiando datos...")

# 1. Relleno de Earnings (Forward Fill)
# Las ganancias se reportan cada 3 meses. El impacto dura hasta el siguiente reporte.
df_final["surprisePercent"] = df_final["surprisePercent"].ffill().fillna(0)

# 2. Relleno de datos técnicos faltantes (VWAP/Trades de Polygon pueden tener huecos)
# Si falta VWAP, usamos Close. Si faltan trades, ponemos 0 o media.
df_final["vwap"] = df_final["vwap"].fillna(df_final["close"])
df_final["num_trades"] = df_final["num_trades"].fillna(0)

# 3. Calcular Indicadores (Features)
# Hacemos esto DESPUÉS de asegurar que no hay huecos en OHLC
df_final = add_derived_features(df_final)

# 4. Limpieza de Nulos generados por indicadores (ej. MA_200 genera 200 NaNs al inicio)
# Eliminamos las filas que no tengan suficientes datos históricos para calcular los indicadores
df_final.dropna(subset=["ma_200", "rsi"], inplace=True)

# 5. Creación del TARGET (Objetivo para ML)
# Target: 1 si el precio sube > 2% en los próximos 5 días
prediction_window = 5
df_final["target_price"] = df_final["close"].shift(-prediction_window)
df_final["target"] = (df_final["target_price"] > df_final["close"] * 1.02).astype(int)

# Eliminamos las ultimas filas donde no hay target (porque hicimos shift negativo)
df_final = df_final.iloc[:-prediction_window]

# D. SELECCIÓN FINAL DE COLUMNAS
cols_finales = [
    "timestamp", "open", "high", "low", "close", "volume", 
    "vwap", "num_trades",                   # De Polygon
    "return", "volatility", "rsi", "ma_50", "ma_200", # Técnicos
    "hl_range", "oc_range", "log_volume", 
    "surprisePercent",                      # Fundamental
    "target"                                # Target
]

# Filtrar solo columnas existentes
cols_existentes = [c for c in cols_finales if c in df_final.columns]
df_final = df_final[cols_existentes]

# E. GUARDADO
output_file = DATA_RAW / f"{ticker}_ml_ready.csv"
df_final.to_csv(output_file, index=False)

print(f"✔ Proceso completado.")
print(f"✔ Datos guardados en: {output_file}")
print(f"✔ Dimensiones finales: {df_final.shape}")
print(df_final.tail())

🚀 Iniciando proceso para NVDA desde 2006-01-10 hasta 2026-01-10...
   ⬇ Descargando YFinance para NVDA...
 Tratando y limpiando datos...
✔ Proceso completado.
✔ Datos guardados en: /home/cacelas/Documentos/Proyects/inversion/data/raw/NVDA_ml_ready.csv
✔ Dimensiones finales: (4828, 18)
Price  timestamp        open        high         low       close     volume  \
5022  2025-12-26  189.919998  192.690002  188.000000  190.529999  139740300   
5023  2025-12-29  187.710007  188.759995  185.910004  188.220001  120006100   
5024  2025-12-30  188.240005  188.990005  186.929993  187.539993   97687300   
5025  2025-12-31  189.570007  190.559998  186.490005  186.500000  120100500   
5026  2026-01-02  189.839996  192.929993  188.259995  188.850006  148240500   

Price        vwap  num_trades    return  volatility        rsi       ma_50  \
5022   190.529999           0  0.010180    0.019795  59.239866  186.058481   
5023   188.220001           0 -0.012124    0.019504  53.096727  186.186883   
5024 

In [ ]:
# -----------------------------
# SAVE
# -----------------------------
output_file = DATA_RAW / f"{ticker}_ml_ready.csv"
df_final.to_csv(output_file, index=False)

print(f"✔ Archivo listo: {output_file}")
print(df_final.tail())


✔ Archivo listo: /home/cacelas/Documentos/Proyects/inversion/data/raw/NVDA_ml_ready.csv
Price  timestamp        open        high         low       close     volume  \
5022  2025-12-26  189.919998  192.690002  188.000000  190.529999  139740300   
5023  2025-12-29  187.710007  188.759995  185.910004  188.220001  120006100   
5024  2025-12-30  188.240005  188.990005  186.929993  187.539993   97687300   
5025  2025-12-31  189.570007  190.559998  186.490005  186.500000  120100500   
5026  2026-01-02  189.839996  192.929993  188.259995  188.850006  148240500   

Price        vwap  num_trades    return  volatility        rsi       ma_50  \
5022   190.529999           0  0.010180    0.019795  59.239866  186.058481   
5023   188.220001           0 -0.012124    0.019504  53.096727  186.186883   
5024   187.539993           0 -0.003613    0.019303  52.973839  186.273487   
5025   186.500000           0 -0.005545    0.019318  53.158385  186.350890   
5026   188.850006           0  0.012601    0.01

In [ ]:
df_final.head()

Price,timestamp,open,high,low,close,volume,vwap,num_trades,return,volatility,rsi,ma_50,ma_200,hl_range,oc_range,log_volume,surprisePercent,target
199,2006-10-24,0.483477,0.502120,0.482102,0.496619,519288000,0.496619,0,0.029133,0.033943,56.920084,0.451737,0.387159,0.020018,-0.013142,20.067969,0.0,1
200,2006-10-25,0.501355,0.508843,0.492952,0.501203,389436000,0.501203,0,0.009231,0.033918,63.429917,0.453961,0.388061,0.015892,0.000152,19.780210,0.0,0
201,2006-10-26,0.502425,0.518775,0.502425,0.517094,366690000,0.517094,0,0.031706,0.033947,64.564537,0.455945,0.388984,0.016350,-0.014669,19.720027,0.0,0
202,2006-10-27,0.517095,0.528555,0.488978,0.494785,566988000,0.494785,0,-0.043143,0.035762,47.145885,0.457613,0.389820,0.039577,0.022309,20.155849,0.0,0
203,2006-10-30,0.498146,0.509607,0.492798,0.500745,479592000,0.500745,0,0.012044,0.035057,53.296685,0.459267,0.390699,0.016808,-0.002598,19.988446,0.0,1
